In [1]:
import ijson
import random
import re
from datetime import datetime

In [2]:
with open("Data/anticipatory_accepted.json", "r", encoding="utf-8") as f:
    objects = ijson.items(f, 'item')
    anticipatory_accepted = list(objects)

with open("Data/anticipatory_denied.json", "r", encoding="utf-8") as f:
    objects = ijson.items(f, 'item')
    anticipatory_denied = list(objects)

with open("Data/regular_accepted.json", "r", encoding="utf-8") as f:
    objects = ijson.items(f, 'item')
    regular_accepted = list(objects)

with open("Data/regular_denied.json", "r", encoding="utf-8") as f:
    objects = ijson.items(f, 'item')
    regular_denied = list(objects)

anticipatory_accepted.sort(key=lambda x: len(x), reverse=True)
anticipatory_denied.sort(key=lambda x: len(x), reverse=True)
regular_accepted.sort(key=lambda x: len(x), reverse=True)
regular_denied.sort(key=lambda x: len(x), reverse=True)

In [3]:
len(anticipatory_accepted), len(anticipatory_denied), len(regular_accepted), len(regular_denied)

(33126, 15078, 29789, 11976)

In [4]:
cutoff = min(len(anticipatory_accepted), len(anticipatory_denied), len(regular_accepted), len(regular_denied))
data = anticipatory_accepted[:cutoff] + anticipatory_denied[:cutoff] + regular_accepted[:cutoff] + regular_denied[:cutoff]
random.shuffle(data)

In [5]:
partial_bail_cases = ['KLHC010007612019',
  'UPHC010021842020',
  'JHHC010044502015',
  'UPHC010509342020',
  'KLHC010023452010',
  'JHHC010297622016',
  'JHHC010228712012',
  'KLHC010028162018',
  'CGHC010288892016',
  'CGHC010288902016',
  'JHHC010300602015',
  'JHHC010182242015',
  'HCBM010025152016',
  'UPHC010898992020',
  'UPHC010022112020',
  'CGHC010288882016',
  'JHHC010116972016',
  'HCBM010210092014']
# these cases have some accused granted bail and some denied bail
# remove the items from data (list) if item['CNR'] is in partial_bail_cases
# open Data/empty_statutes_cnrs.txt and read the CNRs from there too and remove those items from data
with open("Data/empty_statutes_cnrs.txt", "r", encoding="utf-8") as f:
    empty_statutes_cnrs = f.read().splitlines()
partial_bail_cases.extend(empty_statutes_cnrs)
data = [item for item in data if item['CNR'] not in partial_bail_cases]

In [6]:
# # dump empty_statutes_cnrs to a file
# with open("Data/empty_statutes_cnrs.txt", "w", encoding="utf-8") as f:
#     for cnr in empty_statutes_cnrs:
#         f.write(cnr + "\n")

In [7]:

class StatuteExtractor:
    def __init__(self, act_names=None, common_abbreviations=None):
        # Compile possible act names (for strict matching), or use a generic fallback if none provided
        possible_act_names = []
        if act_names:
            for act in act_names:
                cleaned_act = re.sub(r'^\bThe\b\s+', '', act, flags=re.IGNORECASE)
                cleaned_act = re.sub(r',?\s+\d{4}$', '', cleaned_act)
                possible_act_names.append(cleaned_act)
            possible_act_names += act_names
        if common_abbreviations:
            possible_act_names += common_abbreviations
        # Remove empties and dedupe
        possible_act_names = [x for x in set(possible_act_names) if x]
        if possible_act_names:
            law_list_regex = r'(?:' + '|'.join(sorted(map(re.escape, possible_act_names), key=len, reverse=True)) + r')'
        else:
            # fallback to generic match for "Act", "Code", "Law", "Rules", "Regulations" etc.
            law_list_regex = r'[A-Za-z].*?(?:Act|Code|Law|Regulation|Rules|Ordinance|Bill|Amendment|Notification|Order|Guidelines|IPC|CrPC|CPC)'
        # Pattern
        self.pattern = re.compile(
            r'''
            \b
            (?:(?:[Ss]ection|[Aa]rticle|[Ss]chedule)s?|[Uu]/[Ss])            # “Section(s)” or “Article(s)” or “u/s”
            \W+                                                  # separator (spaces, hyphens…)
            (?P<sections>                                        # one or more section‑number tokens
                (?:
                    (?:[Ss]ection|[Aa]rticle|[Ss]chedule\s*)?
                    \d+(?:\s*\(([A-Za-z]{1,2}|[0-9]{1,3})\)|[-_]([A-Za-z]{1,2}|[0-9]{1,3}))?
                )
                (?:
                    \s*(?:,\s*|\s+and\s+)
                    (?:[Ss]ection|[Aa]rticle|[Ss]chedule\s*)?
                    \d+(?:\s*\(([A-Za-z]{1,2}|[0-9]{1,3})\)|[-_]([A-Za-z]{1,2}|[0-9]{1,3}))?
                )*
            )
            \s+
            ,?\s*
            (?:of\s+)?                                          # optional “of”
            (?:the\s+)?                                         # optional “the”
            (?P<law_name>
                (?:Code\s+(?:of|on|for|to)(?:\s+[A-Z][a-zA-Z]*)+)
              |
                (?:[A-Z][a-zA-Z]*(?:\s+[A-Z][a-zA-Z]*)*\s+(?:Act|Code|Law|Regulation|Rules|Ordinance|Bill|Amendment|Notification|Order|Guidelines|Adhiniyam))
              |
                ''' + law_list_regex + r'''
              |
                (?:[A-Za-z]{1,2}\.)+         # one or more “XX.” segments
                (?:[A-Za-z]{1,2}\.?)?        # optional final "XX" with optional dot
            )
            \b
            ''',
            flags=re.IGNORECASE | re.VERBOSE
        )

    def normalize_section_name(self, section):
        section_name = section.lower()  # Normalize to lowercase
        section_name = re.sub(r'[^A-Za-z0-9]', '_', str(section_name))
        section_name = re.sub(r'([A-Za-z])([0-9])', r'\1_\2', section_name)
        section_name = re.sub(r'([0-9])([A-Za-z])', r'\1_\2', section_name)
        section_name = re.sub(r'_+', '_', section_name)
        section_name = section_name.strip('_')
        section_name = re.sub(r'(^_?0\b|(?<=_)0\b|^0_|\b0_?$)', '', section_name)
        section_name = re.sub(r'_+', '_', section_name).strip('_')
        return section_name
    
    def section_correction(self, section):
        # Remove underscore and everything after it if underscore is between digits or followed by digit
        if re.search(r'\d+_\d+', section) or re.search(r'_\d+', section):
            corrected_section = re.sub(r'_\d+.*', '', section)
        else:
            corrected_section = section
        # Convert letters to uppercase and remove all underscores
        corrected_section = re.sub(r'_', '', corrected_section)
        corrected_section = re.sub(r'([a-zA-Z]+)', lambda m: m.group(1).upper(), corrected_section)        
        return corrected_section


    def extract(self, text):
        """Returns a list of {'section': ..., 'act': ...} elements."""
        matches = self.pattern.finditer(text)
        results = []
        seen = set()
        for match in matches:
            sections_chunk = match.group('sections')
            law_name = match.group('law_name').strip()
            if law_name.lower() in ['ipc', 'i.p.c.', 'i.p.c', 'indian penal code', ]:
                law_name = 'IPC'
            if law_name.lower() in ['crpc', 'cr.p.c.','cr.pc.', 'cr.p.c', 'code of criminal procedure', 'criminal procedure code']:
                law_name = 'CrPC'
            if law_name.lower() in ['posco', 'pocso', 'posco act', 'pocso act', 'p.o.s.c.o.', 'p.o.s.c', 'protection of children from sexual offences act']:
                law_name = 'POSCO'
            raw_sections = re.split(r'(?:,|\band\b)', sections_chunk)
            for sect in raw_sections:
                sect = sect.strip()
                if sect:
                    sect = re.sub(r'^[Ss]ection\s*', '', sect)
                    sect = re.sub(r'^[Aa]rticle\s*', '', sect)
                    sect = self.normalize_section_name(sect)
                    sect = self.section_correction(sect)
                    name = f"section_{sect}_of_{law_name}"
                    if name in seen:
                        continue
                    seen.add(name)
                    results.append({'section': sect, 'act': law_name})
        return results

In [8]:
common_abbreviations = [
    'IPC', 'CrPC', 'CPC', 'IT Act', 'RTI Act', 'POSCO', 'SC/ST Act', 'PoA', 'FCRA', 'FEMA', 'PMLA', 'NDPS', 'HMA', 'DV Act',
    'DVC', 'NIA', 'RERA', 'DRT', 'CGST Act', 'SEBI Act', 'MVA', 'ESMA', 'POTA', 'PoTA', 'COFEPOSA', 'CoFEPoSA', 'CAT', 'PSA',
    'UAPA', 'NSA', 'MCOCA', 'SARFAESI', 'SARFAESI Act', 'JJ Act', 'SHWW Act', 'POSH Act', 'SHWW', 'POSH', 'PoSH', 'FEOA', 'GI Act',
    'PCA', 'NDPS', 'NDPS Act', 'Dowry Act', 'Dowry Proh. Act', 'IEA', 'UPPRA', 'UPPCA', 'Cow Slaughter Act', 'UPPDA', 'UPPFA', 'Goonda Act', 'Gangsters Act',
]

In [9]:
MISSING_PAT = re.compile(
    r"\b(not\s+provided|unknown|not\s+known|na|n/a|none|null|missing)\b",
    flags=re.IGNORECASE
)

def parse_age_field(age_text: str, min_age=0, max_age=120):
    """
    Parses a raw age string extracted from judgement text.

    Returns:
      age_given (bool):
          True if at least one numeric age could be extracted.
          This answers: "Do we have usable numeric age information?"

      ages (list[int]):
          List of all extracted ages (filtered to plausible range).

      missing_flag (bool):
          True if the text explicitly states that age information
          is missing or unknown (e.g. 'not provided', 'unknown').

          NOTE:
          age_given and missing_flag are NOT opposites.
          Both can be True in cases like:
              "38 years, not provided"
          which means some age info is present, but the text admits
          incomplete or partial information.
    """
    if age_text is None:
        return False, [], True

    s = str(age_text).strip().lower()

    # Detect explicit textual signals that age is missing/unknown
    missing_flag = bool(MISSING_PAT.search(s))

    # Extract all numeric substrings (handles '28, 27 years', '36/17 years', etc.)
    nums = [int(x) for x in re.findall(r"\d+", s)]

    # Keep only plausible human ages
    ages = [n for n in nums if min_age <= n <= max_age]

    # Numeric availability flag:
    # True if we can compute an age value from the text at all
    age_given = len(ages) > 0

    return age_given, ages, missing_flag


In [10]:
def get_bail_type(json_obj):
    details = json_obj['case']
    pattern = r"Applicant applied for\s+(\w+\-\w+)\."
    match = re.search(pattern, details)
    if match:
        bail_type = match.group(1).strip()
        return bail_type.lower()
    return None

def get_age(json_obj):
    details = json_obj['case']
    pattern = r'Age(?: of the accused)? is\s+(.+?)\.'
    match = re.search(pattern, details)
    if match:
        age = match.group(1).strip().lower()
        age_given, ages, missing_flag = parse_age_field(age)
        if age_given:
            return (True, ages)
        else:
            return (False, None)
    return (False, None)

def get_health_condition(json_obj):
    details = json_obj['case']
    pattern = r'Health issues for the accused are\s+(.+?)\n'
    match = re.search(pattern, details, flags=re.IGNORECASE)
    if match:
        health_condition = match.group(1).strip()
        return health_condition.lower()
    return None

def get_past_criminal_record(json_obj):
    details = json_obj['case']
    NEGATION_WORDS = {"no", "none", "not", "nil", "without"}
    pattern = r'There are\s+(.*?)past criminal records of the accused(?:\s*\[(.*?)\])?\.\n'
    match = re.search(pattern, details, flags=re.IGNORECASE)
    if match:
        existence_part = match.group(1).lower().strip()
        charges_part = match.group(2)
        if any(neg_word in existence_part for neg_word in NEGATION_WORDS):
            return (False, None)
        if charges_part is None or any(neg_word in charges_part.lower() for neg_word in NEGATION_WORDS):
            return (True, None)
        cm = re.search(r"charges\s*:\s*(.*)$", charges_part, flags=re.IGNORECASE)
        charges_str = cm.group(1).strip() if cm else charges_part.strip()
        extractor = StatuteExtractor(common_abbreviations=common_abbreviations)
        charges = extractor.extract(charges_str)
        past_statutes = [f"{charge['section']} {charge['act']}" for charge in charges]
        return (True, past_statutes)
        
    return (None, None)

def get_case_details(json_obj):
    details = json_obj['case']
    pattern = r'Details of the incident are\s*(.*)$'    
    match = re.search(pattern, details, flags=re.IGNORECASE | re.DOTALL)
    if match:
        case_details = match.group(1).strip()
        return case_details
    return None

def get_days_in_custody(json_obj):
    date_of_arrest = None if json_obj['date_of_arrest'] in [None, '', 'Unknown'] else json_obj['date_of_arrest']
    date_of_judgement = None if json_obj['date_of_judgement'] in [None, '', 'Unknown'] else json_obj['date_of_judgement']
    if date_of_arrest is not None and date_of_judgement is not None:
        fmt = "%d-%m-%Y"
        try:
            arrest_date = datetime.strptime(date_of_arrest, fmt)
            judgement_date = datetime.strptime(date_of_judgement, fmt)
            delta = judgement_date - arrest_date
            return abs(delta.days)
        except:
            return None
    return None

def get_outcome(json_obj):
    details = json_obj['outcome']
    pattern = r'The outcome of the case is\s+(.+?)\.'    
    match = re.search(pattern, details)
    if match:
        outcome = match.group(1).strip()
        return outcome.lower()
    return None

In [11]:
past_crimes = set()
for item in data:
    has_past_crime, charges = get_past_criminal_record(item)
    if has_past_crime:
        if charges is not None:
            for charge in charges:
                past_crimes.add(charge)



In [12]:
past_crimes

{'10 Gunda Act',
 '10 IPC',
 '10 UAP Act',
 '107 CrPC',
 '107 the Code',
 '109 CrPC',
 '109 IPC',
 '11 Animal Cruelty Act',
 '11 Maharashtra Animal Preservation Act',
 '11 POSCO',
 '11 Prevention of Cruelty to Animals Act',
 '11 Wildlife Protection Act',
 '110 CrPC',
 '110G CrPC',
 '112 IPC',
 '114 IPC',
 '115 IPC',
 '117E KP Act',
 '118 IPC',
 '118 Kerala Police Act',
 '118A KP Act',
 '118A Kerala Police Act',
 '118E KP Act',
 '119A Kerala Police Act',
 '12 Indian Passport Act',
 '12 POSCO',
 '120 IPC',
 '120B IPC',
 '120O Kerala Police Act',
 '121 IPC',
 '121A IPC',
 '122 IPC',
 '12V Kerala Protection of River Banks and Regulation of Removal of Sand Act',
 '13 Criminal Law Amendment Act',
 '13 IPC',
 '13 Indian Passport Act',
 '13 Kerala Money Lenders Act',
 '13 Money Lenders Act',
 '13 U.A.P',
 '13 UAP Act',
 '13 Unlawful Activities Prevention Act',
 '135 Bombay Police Act',
 '135 Indian Electricity Act',
 '135 Maharashtra Police Act',
 '135 Mumbai Police Act',
 '137 Maharashtra Pol

In [13]:
aug_data = []
for item in data:
    obj = {}
    obj['CNR'] = item['CNR']
    obj['bail_type'] = get_bail_type(item)
    obj['age_available'], obj['ages'] = get_age(item)
    obj['health_condition'] = get_health_condition(item)
    obj['past_criminal_record_exists'], obj['past_criminal_record_charges'] = get_past_criminal_record(item)
    obj['statutes'] = list(set(item['statutes']))
    obj['case_details'] = get_case_details(item)
    obj['days_in_custody'] = get_days_in_custody(item)
    obj['outcome'] = get_outcome(item)
    obj['reasoning'] = item['reasoning']
    obj['statute_details'] = list(set(item['context']))
    aug_data.append(obj)

In [14]:
import json

with open('Data/data_augmented.json', 'w', encoding='utf-8') as f:
    json.dump(aug_data, f, indent=4, ensure_ascii=False)

In [15]:
aug_data[:5]

[{'CNR': 'KLHC010039992013',
  'bail_type': 'anticipatory-bail',
  'age_available': False,
  'ages': None,
  'health_condition': 'none.',
  'past_criminal_record_exists': False,
  'past_criminal_record_charges': None,
  'statutes': ['143 IPC',
   '323 IPC',
   '341 IPC',
   '147 IPC',
   '324 IPC',
   '149 IPC',
   '148 IPC',
   '438 CrPC',
   '326 IPC'],
  'case_details': "The allegation against the petitioners is that they, out of their previous enmity towards the de facto complainant and the other workers belong to the INTUC Union at the IOC Plant, formed themselves into an unlawful assembly, armed with deadly weapons, on the road in front of the IOC Plant and wrongfully restrained the de facto complainant and other workers of the Union, and the first accused beat the de facto complainant on his right hand with an iron pipe, thereby causing fracture of his ulna. It is alleged that the 3rd accused attacked one Jaimon, who was along with the de facto complainant, with the paper cuttin

In [16]:
sample = []
count = 0
for item in data:
    details = item['case']
    if 'There are no past criminal records of the accused.' not in details:
        for k,v in item.items():
            print(k, ":", v)
        print("\n")
        count += 1
        sample.append(item)
        if count >= 5:
            break

CNR : HCBM010132752016
case : Applicant applied for Regular-Bail.
Is it a withdrawal application? No.
Age of the accused is not provided.
Health issues for the accused are None.
There are some past criminal records of the accused [charges: Section 302 IPC].
Statutes against the accused in the case are [Section 387 IPC, Section 504 IPC, Section 506(2) IPC, Section 34 IPC, Section 3(25) Indian Arms Act].
Precedents mentioned in the judgement are None.
Details of the incident are The applicant-accused is facing charges under sections 387, 504, 506(2) r/w section 34 of the Indian Penal Code and section 3(25) of the Indian Arms Act in C.R. No.247 of 1998 registered with the Bhayander police station.
Arguments supporting the bail application are The learned Counsel for the Applicant has submitted that the applicant-accused is inside under the said C.R. for nearly 5 years and the maximum punishment under section 387 is 7 years. However, he was absconding in between. Though he is in custody of

In [17]:
for item in sample:
    print(get_past_criminal_record(item))

(True, ['302 IPC'])
(True, None)
(True, [])
(True, [])
(True, ['376 IPC', '354 IPC', '342 IPC', '294 IPC', '323 IPC', '506 IPC'])


In [18]:
data[3]

{'CNR': 'KLHC010089182013',
 'case': 'Applicant applied for Regular-Bail.\nIs it a withdrawal application? No.\nAge of the accused is 20 years.\nHealth issues for the accused are None.\nThere are no past criminal records of the accused.\nStatutes mentioned in the judgement are [Section 306 IPC, Section 420 IPC].\nPrecedents mentioned in the judgement are None.\nDetails of the incident are In respect of an incident which took place on 04.01.2013 whereby a lady aged 18 years committed suicide, the petitioner was later implicated as accused for having committed the offences punishable under Sections 306 and 420 IPC. The allegation was that he had managed to obtain the gold ornaments of the lady and sell the same and when she asked for the same, he failed to return.\nArguments supporting the bail application are The petitioner would say that he is innocent. According to him, he and the victim were in love and their relationship was not to the liking of the parents of the girl and he has be

In [19]:
print(get_health_condition(data[3]))

none.


In [20]:
data_aug = []



In [21]:
# create a train-val-test split
train_data = data[:int(0.8*len(data))]
val_data = data[int(0.8*len(data)):int(0.9*len(data))]
test_data = data[int(0.9*len(data)):]

In [22]:
len(train_data), len(val_data), len(test_data)

(36914, 4614, 4615)

In [2]:
import json
with open('Data/data_augmented.json', 'r', encoding='utf-8') as f:
    aug_data = json.load(f)

print(len(aug_data))

46143
